# PWV05 : Seasonal Variation Transparency Simulation — Publication-quality figures

**Goal :** Demonstrate annual PWV variations on transparency at Cerro Pachon with

**Author :** Sylvie Dagoret-Campagne — IJCLab/IN2P3/CNRS  
**Created :** 2026-03-29  
**Last update:** 2026-03-30 :
**Kernel :** conda_py313

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
warnings.resetwarnings()
warnings.simplefilter('ignore')

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator, AutoMinorLocator
from scipy.optimize import curve_fit
from astropy.time import Time

%matplotlib inline

# ── publication-style rc params ───────────────────────────────────────────────
mpl.rcParams.update({
    'figure.dpi'     : 150,
    'font.family'    : 'serif',
    'font.size'      : 11,
    'axes.labelsize' : 13,
    'axes.titlesize' : 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'axes.grid'      : True,
    'grid.alpha'     : 0.35,
    'axes.linewidth' : 1.1,
})

In [ ]:
from PWV00_parameters import (
    version_run, tag, extractedfilesdict, legendtag,
    PWV_FILTER_LIST, FLAG_PWVFILTERS,
    DumpConfig
)
from mysitcom.auxtel.qualitycuts import ParameterCutSelection, ParameterCutTools
from mysitcom.auxtel.pwv import normalize_column_data_bytarget_byfilter

DumpConfig()

In [ ]:
# ── output directories ────────────────────────────────────────────────────────
pathfigs = "figs_PWV05seas"
prefix   = "pwv05seas"
os.makedirs(pathfigs, exist_ok=True)
figtype  = ".pdf"   # vector for journal submission; change to .png for quick preview

In [ ]:
#Cut bad PWV from Merra2
PWVMIN = 0.
PWVMAX = 16.

---
## 1. Load data

In [ ]:
version_run

In [ ]:
sys.modules['numpy.rec'] = np.rec

In [ ]:
atmfilename   = extractedfilesdict[version_run]
inputfilename = atmfilename.split("/")[-1]
print(f"Loading: {atmfilename}")

if "parquet" in inputfilename:
    df_spec = pd.read_parquet(atmfilename)
else:
    specdata = np.load(atmfilename, allow_pickle=True)
    df_spec  = pd.DataFrame(specdata)

# ── Build Time column from DATE-OBS, exactly as in PWV01 ─────────────────────
df_spec["Time"] = pd.to_datetime(df_spec["DATE-OBS"], utc=True)
df_spec = df_spec.sort_values(by="Time", ascending=True).reset_index(drop=True)

print(f"Shape: {df_spec.shape}")
print(f"Time range: {df_spec['Time'].min()}  →  {df_spec['Time'].max()}")
print("Columns:", ", ".join(df_spec.columns))

In [ ]:
# ── rename Corentin run columns to standardised names ─────────────────────────
if "run2026_v02" in version_run:
    df_spec["chi2_ram"] = df_spec["CHI2_FIT"]
    df_spec["chi2_rum"] = df_spec["CHI2_FIT"]

# ── keep only PWV-sensitive filters ───────────────────────────────────────────
if FLAG_PWVFILTERS:
    df_spec = df_spec[df_spec["FILTER"].isin(PWV_FILTER_LIST)].copy()

# ── drop rows with missing PWV ────────────────────────────────────────────────
pwv_cols = ["PWV [mm]_ram", "PWV [mm]_rum", "PWV [mm]_err_ram", "PWV [mm]_err_rum"]
df_spec.dropna(subset=pwv_cols, inplace=True)

print(f"After filter selection: {len(df_spec)} rows")
print("Filters present:", df_spec["FILTER"].unique())

In [ ]:
# ── normalise chi2 per target per filter (needed for quality cuts) ─────────────
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col="TARGET", filter_col="FILTER",
    feature_col="CHI2_FIT", ext="norm")
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col="TARGET", filter_col="FILTER",
    feature_col="chi2_ram", ext="norm")
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col="TARGET", filter_col="FILTER",
    feature_col="chi2_rum", ext="norm")

---
## 2. Apply quality cuts (tight)

In [ ]:
FLAG_LOOSE_CUTS = True
FLAG_TIGHT_CUTS = False

In [ ]:
pathdata      = "data_PWV01seas"   # reuse cuts from PWV01 notebook
if FLAG_LOOSE_CUTS:
    filename_cuts = f"{pathdata}/cuts_loose_finaldecision.json" 
    cutstype_name = "loose-cuts"
elif FLAG_TIGHT_CUTS: 
    filename_cuts = f"{pathdata}/cuts_tight_finaldecision.json" 
    cutstype_name = "tight-cuts"
else:
    filename_cuts = f"{pathdata}/cuts_finaldecision.json" 
    cutstype_name = "standard-cuts"

cuts           = ParameterCutTools.load_cuts_json(filename_cuts)
list_of_params = list(cuts.keys())

selector    = ParameterCutSelection(df_spec, params=list_of_params, id_col="id")
flags       = selector.apply_cuts(cuts)
df_selected = df_spec.merge(flags, on="id")
df_keep     = df_selected[df_selected["pass_all_cuts"]].copy()

print(f"Tight cuts: {len(df_keep)} / {len(df_spec)} observations kept")

---
## 3. Derived time columns

`Time` (UTC-aware datetime) was built from `DATE-OBS` in §1, exactly
as in PWV01.  Here we derive calendar fields and the MJD fractional-year
column used for the sinusoidal fit.

In [ ]:
# ── constants ─────────────────────────────────────────────────────────────────
MJD_J2000 = 51544.5   # 2000-01-01T12:00 UTC
YEAR_DAYS  = 365.25

# ── calendar fields from the existing Time column ─────────────────────────────
df_keep["year"]     = df_keep["Time"].dt.year
df_keep["month"]    = df_keep["Time"].dt.month
df_keep["doy"]      = df_keep["Time"].dt.day_of_year   # 1-366
df_keep["doy_norm"] = (df_keep["doy"] - 1) / 365.25    # 0→1  Jan→Dec

# ── fractional year since J2000 (sine-fit x-axis) ────────────────────────────
df_keep["mjd_year"] = (df_keep["MJD"] - MJD_J2000) / YEAR_DAYS

mjd_min = df_keep["MJD"].min()
mjd_max = df_keep["MJD"].max()

# ── best-estimate PWV and combined uncertainty ────────────────────────────────
df_keep["PWV"]     = 0.5 * (df_keep["PWV [mm]_ram"] + df_keep["PWV [mm]_rum"])
df_keep["PWV_err"] = np.sqrt(
    df_keep["PWV [mm]_err_ram"]**2 + df_keep["PWV [mm]_err_rum"]**2
)

years_present = sorted(df_keep["year"].dropna().astype(int).unique())
print("Years in dataset :", years_present)
print(f"MJD range        : {mjd_min:.1f} — {mjd_max:.1f}")
print(f"Date range       : {df_keep['Time'].min().date()}  →  {df_keep['Time'].max().date()}")

In [ ]:
MONTH_ABBR = ["Jan","Feb","Mar","Apr","May","Jun",
              "Jul","Aug","Sep","Oct","Nov","Dec"]

---
## 5. Monthly weighted statistics

In [ ]:
monthly = []
for m in range(1, 13):
    sub = df_keep[df_keep["month"] == m]
    if len(sub) == 0:
        monthly.append({"month": m, "pwv_mean": np.nan, "pwv_sem": np.nan, "n": 0})
        continue
    w   = 1.0 / (sub["PWV_err"].values**2 + 1e-12)
    mu  = np.average(sub["PWV"].values, weights=w)
    sem = 1.0 / np.sqrt(w.sum())
    monthly.append({"month": m, "pwv_mean": mu, "pwv_sem": sem, "n": len(sub)})

df_monthly = pd.DataFrame(monthly)
display(df_monthly)

---
## 12. Seasonal PWV statistics

Seasons defined for the **Southern Hemisphere** (Cerro Pachon):

| Season | Months |
|--------|--------|
| Summer (DJF) | December, January, February |
| Autumn (MAM) | March, April, May |
| Winter (JJA) | June, July, August |
| Spring (SON) | September, October, November |

Statistics computed over **all years** and **all PWV-sensitive filters**
after tight quality cuts:
- **N** : number of individual measurements
- **Mean** : arithmetic mean [mm]
- **Median** : median [mm]
- **Std** : standard deviation [mm]
- **Wt. mean** : weighted mean (weights = 1/sigma^2) [mm]
- **Wt. std** : weighted standard deviation [mm]

In [ ]:
# ── Season definition — Southern Hemisphere ───────────────────────────────────
#SEASON_MAP = {
#    12: "Summer (DJF)",  1: "Summer (DJF)",  2: "Summer (DJF)",
#     3: "Autumn (MAM)",  4: "Autumn (MAM)",  5: "Autumn (MAM)",
#     6: "Winter (JJA)",  7: "Winter (JJA)",  8: "Winter (JJA)",
#     9: "Spring (SON)", 10: "Spring (SON)", 11: "Spring (SON)",
#}
SEASON_MAP = {
    12: "Summer",  1: "Summer",  2: "Summer",
     3: "Autumn",  4: "Autumn",  5: "Autumn",
     6: "Winter",  7: "Winter",  8: "Winter",
     9: "Spring", 10: "Spring", 11: "Spring",
}
#SEASON_ORDER = ["Summer (DJF)", "Autumn (MAM)", "Winter (JJA)", "Spring (SON)"]
SEASON_ORDER = ["Summer", "Autumn", "Winter", "Spring"]
SEASON_COLORS = ["red", "orange","blue","green"]
df_keep["season"] = df_keep["month"].map(SEASON_MAP)


def weighted_stats(vals, errs):
    """Weighted mean and weighted standard deviation."""
    w     = 1.0 / (errs**2 + 1e-12)
    mu_w  = np.average(vals, weights=w)
    std_w = np.sqrt(np.average((vals - mu_w)**2, weights=w))
    return mu_w, std_w


rows = []
for season in SEASON_ORDER:
    sub  = df_keep[df_keep["season"] == season]
    vals = sub["PWV"].values
    errs = sub["PWV_err"].values
    mask = np.isfinite(vals) & np.isfinite(errs) & (errs > 0)
    vals, errs = vals[mask], errs[mask]
    if len(vals) == 0:
        rows.append({"Season": season, "N": 0,
                     "Mean [mm]": np.nan, "Median [mm]": np.nan,
                     "Std [mm]": np.nan,
                     "Wt. mean [mm]": np.nan, "Wt. std [mm]": np.nan})
        continue
    mu_w, std_w = weighted_stats(vals, errs)
    rows.append({
        "Season"       : season,
        "N"            : len(vals),
        "Mean [mm]"    : np.mean(vals),
        "Median [mm]"  : np.median(vals),
        "Std [mm]"     : np.std(vals, ddof=1),
        "Wt. mean [mm]": mu_w,
        "Wt. std [mm]" : std_w,
    })

df_seasons = pd.DataFrame(rows).set_index("Season")

# ── styled display in notebook ────────────────────────────────────────────────
fmt_dict = {c: "{:.3f}" for c in df_seasons.columns if c != "N"}
fmt_dict["N"] = "{:d}"

display(
    df_seasons.style
    .format(fmt_dict)
    .set_caption(
        f"Seasonal PWV statistics — {version_run} (tight cuts, Southern Hemisphere)"
    )
    .set_table_styles([
        {"selector": "caption",
         "props": [("font-weight", "bold"), ("font-size", "13px"),
                   ("text-align", "left"), ("padding-bottom", "8px")]},
        {"selector": "th",
         "props": [("background-color", "#dce6f1"), ("font-weight", "bold"),
                   ("text-align", "center")]},
        {"selector": "td",
         "props": [("text-align", "right"), ("padding", "4px 10px")]},
        {"selector": "tr:nth-child(even)",
         "props": [("background-color", "#f5f8fc")]},
        {"selector": "tr:hover",
         "props": [("background-color", "#fff3cd")]},
    ])
    .bar(subset=["Mean [mm]", "Median [mm]", "Wt. mean [mm]"],
         color="#aec6e8", vmin=0)
)

In [ ]:
df_keep

In [ ]:
print(list(df_keep.columns))

## Select the filter

In [ ]:
df_keep = df_keep[df_keep["FILTER"] == "empty" ]

In [ ]:
df = df_keep[["FILTER","season","VAOD_ram","ozone [db]_ram","PWV [mm]_ram"]]

In [ ]:
df

In [ ]:
fig,axs = plt.subplots(1,2,figsize=(16,5))
ax1,ax2 = axs.flatten()
df["VAOD_ram"].hist(bins=100,range=(0,1.),ax=ax1)
df["ozone [db]_ram"].hist(bins=100,range=(0,600.),ax=ax2)

In [ ]:
cut_sel = (df["VAOD_ram"] < 0.1) & (df["ozone [db]_ram"] > 20)
df = df[cut_sel]

## Simulate Transmission

In [ ]:
df = df.rename(columns={
    "PWV [mm]_ram": "PWV",
    "ozone [db]_ram": "ozone",
    "VAOD_ram":"VAOD"
    
})

In [ ]:
from getObsAtmo.getObsAtmo import ObsAtmo,validateObsName,Dict_Of_sitesPressures,get_obssite_dataframe

In [ ]:
emul =  ObsAtmo(obs_str= "LSST")
WL = emul.GetWL()
am0=1

In [ ]:
season_to_color = {}
for season, color in zip(SEASON_ORDER, SEASON_COLORS):
    season_to_color[season] = color
fig,ax = plt.subplots(1,1,figsize=(8,6))
for idx, row in df.iterrows():
    season  = row["season"]
    color = season_to_color[season]
    pwv = row["PWV"]
    oz = row["ozone"]
    tau = row["VAOD"] 	
    
    transm = emul.GetAllTransparencies(WL,am0,pwv,oz,tau)
    ax.plot(WL,transm,color=color)
plt.show()    

In [ ]:
season_to_color = dict(zip(SEASON_ORDER, SEASON_COLORS))

fig, ax = plt.subplots(figsize=(8,6))

for season in SEASON_ORDER:
    sub = df[df["season"] == season]
    
    curves = []
    
    for row in sub.itertuples(index=False):
        transm = emul.GetAllTransparencies(
            WL,
            am0,
            row._asdict()["PWV"],
            row._asdict()["ozone"],
            row.VAOD
        )
        curves.append(transm)
    
    curves = np.array(curves)  # shape (N, len(WL))
    
    median = np.median(curves, axis=0)
    p16 = np.percentile(curves, 16, axis=0)
    p84 = np.percentile(curves, 84, axis=0)
    
    color = season_to_color[season]
    
    ax.plot(WL, median, color=color, label=season)
    ax.fill_between(WL, p16, p84, color=color, alpha=0.3)

ax.set_xlabel("Wavelength")
ax.set_ylabel("Transmission")
ax.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm

season_to_cmap = {
    "Spring": cm.Greens,
    "Summer": cm.Reds,
    "Autumn": cm.Oranges,
    "Winter": cm.Blues,
}

In [ ]:
import numpy as np

fig, ax = plt.subplots(1,1,figsize=(6,4),layout="constrained")

for season in ["Summer","Winter"]:
    sub = df[df["season"] == season]
    
    # --- variable physique clé ---
    pwv = sub["PWV"].values
    
    # --- percentiles (0 → 1) ---
    ranks = pwv.argsort().argsort()
    percentiles = ranks / (len(pwv) - 1)
    
    cmap = season_to_cmap[season]
    
    for row, p in zip(sub.itertuples(index=False), percentiles):
        
        # 🔑 inversion pour avoir médiane sombre
        intensity = 1 - abs(p - 0.5) * 2  
        # → 1 à la médiane, 0 aux extrêmes
        
        color = cmap(intensity)
        
        transm = emul.GetAllTransparencies(
            WL,
            am0,
            row._asdict()["PWV"],  # ou version renommée propre
            row._asdict()["ozone"],
            row.VAOD
        )
        
        ax.plot(WL, transm, color=color,lw=1 ,alpha=1)


        
ax.set_xlabel("Wavelength")
ax.set_ylabel("Transmission")
plt.show()


In [ ]:
from matplotlib.colors import TwoSlopeNorm

norm = TwoSlopeNorm(vmin=pwv.min(), vcenter=np.median(pwv), vmax=pwv.max())

color = cmap(norm(row.PWV))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde, rankdata
import matplotlib.cm as cm

season_to_cmap = {
    "Spring": cm.Greens,
    "Summer": cm.Reds,
    "Autumn": cm.Oranges,
    "Winter": cm.Blues,
}

In [ ]:
def plot_season_density(ax, sub, season):

    season_to_color = dict(zip(SEASON_ORDER, SEASON_COLORS))

    # --- construire les courbes ---
    curves = np.array([
        emul.GetAllTransparencies(
            WL,
            am0,
            row["PWV"],
            row["ozone"],
            row["VAOD"]
        )
        for _, row in sub.iterrows()
    ])

    # --- grille transmission ---
    t_min, t_max = curves.min(), curves.max()
    t_grid = np.linspace(t_min, t_max, 150)

    density = np.zeros((len(t_grid), len(WL)))

    # --- KDE ---
    for i in range(len(WL)):
        kde = gaussian_kde(curves[:, i])
        density[:, i] = kde(t_grid)

    # --- fond densité ---
    ax.pcolormesh(
        WL,
        t_grid,
        density,
        shading="auto",
        cmap="Greys",
        alpha=0.4
    )

    # --- percentile PWV ---
    pwv = sub["PWV"].values
    percentiles = rankdata(pwv) / len(pwv)

    cmap = season_to_cmap[season]

    # --- overlay de quelques courbes (sinon trop dense) ---
    idx_sample = np.linspace(0, len(sub)-1, min(50, len(sub))).astype(int)

    for i in idx_sample:
        row = sub.iloc[i]
        p = percentiles[i]

        intensity = 1 - abs(p - 0.5) * 2  # médiane sombre

        color = cmap(intensity)

        transm = curves[i]

        ax.plot(WL, transm, color=color, alpha=0.8, lw=1)

    # --- médiane ---
    median = np.median(curves, axis=0)
    ax.plot(WL, median, color=season_to_color[season], lw=1,alpha=1)

    ax.set_title(season)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12,8), sharex=True, sharey=True)

for ax, season in zip(axes.flat, SEASON_ORDER):
    sub = df[df["season"] == season]
    plot_season_density(ax, sub, season)

for ax in axes[-1]:
    ax.set_xlabel("Wavelength")

for ax in axes[:,0]:
    ax.set_ylabel("Transmission")

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6,4))

for season in ["Summer","Winter"]:
    sub = df[df["season"] == season]
    plot_season_density(ax, sub, season)


ax.set_xlabel("Wavelength")
ax.set_ylabel("Transmission")
ax.set_title("Transmission in Winter and Summer")

plt.tight_layout()
plt.show()